# ConvLSTM — обучение на клипах IRT-кадров

**ConvLSTM SegNet** — сегментация по **последовательности сырых термокадров** (не TSR/Fourier).

| | |
|---|---|
| Вход | `(T, 1, H, W)` — клип из `num_frames` кадров |
| Модель | CNN-encoder → ConvLSTM @ H/4 → decoder + skip |
| Выход | одна маска `(1, H, W)` после просмотра всего клипа |
| Данные | kaggle `R_/Z_` + TPU `sample*` (`dataset_convlstm.yaml`) |
| Loss | BCE + Dice, `pos_weight=10` |

Ноутбук вызывает тот же `train_one()`, что `segmentation/ConvLSTM/train.py`, с **live-графиками** после каждой эпохи.

> Перед первым запуском: `python scripts/build_irt_cache.py --yaml segmentation/ConvLSTM/dataset_convlstm.yaml`  
> На MPS: `NUM_WORKERS=0`, `COMPILE=False`.


In [ ]:
# === параметры (как train.py) ===
EPOCHS = 30
BATCH_SIZE = 4              # None → из yaml (4)
TEST_EVERY = 4
NUM_WORKERS = 0               # MPS: 0
LR = 3e-4
WEIGHT_DECAY = 1e-4
POS_WEIGHT = 10.0
COMPILE = False
DEVICE = "auto"
RESUME = None                 # None | "best" | "last" | Path к .tar
YAML = None                   # None → dataset_convlstm.yaml
PREVIEW_EVERY = 5             # превью предсказания каждые N эпох (0 = только в конце)


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import clear_output, display


def find_project_root() -> Path:
    start = Path.cwd().resolve()
    candidates = [start]
    if start.name == "notebooks":
        candidates.append(start.parent)
    seen: set[Path] = set()
    for base in candidates:
        for p in [base, *base.parents]:
            p = p.resolve()
            if p in seen:
                continue
            seen.add(p)
            if (p / "data").is_dir() and (p / "irt_data").is_dir():
                return p
    raise FileNotFoundError(f"Не найден корень репо. cwd={start}")


ROOT = find_project_root()
CLSTM = ROOT / "segmentation" / "ConvLSTM"
SEG = ROOT / "segmentation"
for p in (CLSTM, SEG, ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from common.device import get_device
from common.mps_train import setup_mps_env, suggest_num_workers
from data import build_temporal_loaders
from train import train_one

setup_mps_env()
%matplotlib inline

device = get_device(DEVICE)
yaml_path = CLSTM / "dataset_convlstm.yaml" if YAML is None else Path(YAML)
if not yaml_path.is_absolute():
    yaml_path = (ROOT / yaml_path).resolve()
nw = suggest_num_workers(device) if NUM_WORKERS is None else NUM_WORKERS
bs = BATCH_SIZE

print(f"ROOT={ROOT}")
print(f"device={device}  yaml={yaml_path.relative_to(ROOT)}")
print(f"bs={bs}  workers={nw}  epochs={EPOCHS}  compile={COMPILE}")


In [ ]:
train_loader, test_loader, train_ds, test_ds = build_temporal_loaders(
    yaml_path,
    test_every=TEST_EVERY,
    batch_size=bs,
    num_workers=nw,
)
T, in_ch, H, W = train_ds[0][0].shape
print(
    f"train: {len(train_ds.video_ids)} videos / {len(train_ds)} samples\n"
    f"test:  {len(test_ds.video_ids)} videos / {len(test_ds)} samples\n"
    f"clip: T={T}  in_ch={in_ch}  size={H}x{W}"
)
print("train ids (first 5):", train_ds.video_ids[:5])


## Просмотр клипа из датасета

In [ ]:
SPLIT = "train"   # "train" | "test"
INDEX = 0


def show_temporal_sample(ds, idx: int, *, title: str = "") -> None:
    frames, mask = ds[idx]
    T = frames.shape[0]
    mask_np = mask.squeeze().numpy()
    pos = float(mask_np.mean())

    # полоска кадров
    cols = min(6, T)
    rows = (T + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(2.2 * cols, 2.2 * rows))
    axes = np.atleast_2d(axes)
    for t in range(rows * cols):
        r, c = divmod(t, cols)
        ax = axes[r, c]
        if t < T:
            ax.imshow(frames[t, 0].numpy(), cmap="inferno")
            ax.set_title(f"t={t}", fontsize=9)
        ax.axis("off")
    fig.suptitle(title or f"clip T={T}", fontsize=12)
    fig.tight_layout()
    plt.show()

    fig, ax = plt.subplots(1, 3, figsize=(10, 3.2))
    mid = T // 2
    ax[0].imshow(frames[mid, 0].numpy(), cmap="inferno")
    ax[0].set_title(f"frame t={mid}")
    ax[1].imshow(mask_np, cmap="gray", vmin=0, vmax=1)
    ax[1].set_title(f"mask pos={pos:.4f}")
    ax[2].imshow(frames[mid, 0].numpy(), cmap="inferno")
    ax[2].contour(mask_np, levels=[0.5], colors="cyan", linewidths=1)
    ax[2].set_title("overlay")
    for a in ax:
        a.axis("off")
    fig.tight_layout()
    plt.show()


ds_view = train_ds if SPLIT == "train" else test_ds
show_temporal_sample(ds_view, INDEX, title=f"{SPLIT}[{INDEX}]")


## Обучение с live-графиками

In [ ]:
def plot_history_live(history: list[dict], *, title: str = "") -> None:
    if not history:
        return
    epochs = [float(r["epoch"]) for r in history]
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
    fig.suptitle(title or "ConvLSTM", fontsize=12)
    specs = [
        ("train_loss", "test_loss", "loss"),
        ("train_iou", "test_iou", "IoU"),
        ("train_dice", "test_dice", "Dice"),
    ]
    for ax, (tk, vk, ylab) in zip(axes, specs):
        ax.plot(epochs, [float(r[tk]) for r in history], "o-", label="train", ms=3)
        ax.plot(epochs, [float(r[vk]) for r in history], "o-", label="test", ms=3)
        ax.set_title(ylab)
        ax.set_xlabel("epoch")
        ax.legend()
        ax.grid(alpha=0.3)
    fig.tight_layout()
    display(fig)
    plt.close(fig)


def preview_prediction(model, ds, device, *, idx: int = 0, title: str = "") -> None:
    model.eval()
    frames, mask = ds[idx]
    T = frames.shape[0]
    with torch.no_grad():
        logits = model(frames.unsqueeze(0).to(device))
        prob = torch.sigmoid(logits).squeeze().cpu().numpy()
    pred = (prob > 0.5).astype(np.float32)
    gt = mask.squeeze().numpy()
    mid = T // 2

    fig, ax = plt.subplots(1, 5, figsize=(14, 3.2))
    if title:
        fig.suptitle(title, fontsize=11)
    ax[0].imshow(frames[mid, 0].numpy(), cmap="inferno")
    ax[0].set_title(f"frame t={mid}")
    ax[1].imshow(gt, cmap="gray", vmin=0, vmax=1)
    ax[1].set_title("GT")
    ax[2].imshow(prob, cmap="magma", vmin=0, vmax=1)
    ax[2].set_title("prob")
    ax[3].imshow(pred, cmap="gray", vmin=0, vmax=1)
    ax[3].set_title("pred")
    ax[4].imshow(frames[mid, 0].numpy(), cmap="inferno")
    ax[4].contour(pred, levels=[0.5], colors="lime", linewidths=0.8)
    ax[4].set_title("overlay")
    for a in ax:
        a.axis("off")
    fig.tight_layout()
    display(fig)
    plt.close(fig)


def _on_epoch_end(tracker, row, epoch, end_epoch, best_iou, best_epoch, opt, model):
    clear_output(wait=True)
    print(tracker.format_line(row, end_epoch))
    print(f"lr={opt.param_groups[0]['lr']:.2e}  best test IoU {best_iou:.4f} @ {best_epoch}")
    plot_history_live(tracker.history, title=f"ConvLSTM — epoch {epoch}/{end_epoch}")
    if PREVIEW_EVERY and epoch % PREVIEW_EVERY == 0:
        preview_prediction(model, test_ds, device, idx=0, title=f"preview @ ep {epoch}")


In [ ]:
tracker, model, best_iou, best_epoch = train_one(
    yaml_path,
    EPOCHS,
    device,
    test_every=TEST_EVERY,
    batch_size=bs,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    pos_weight=POS_WEIGHT,
    compile_model=COMPILE,
    num_workers=nw,
    resume=RESUME,
    on_epoch_end=_on_epoch_end,
)

print(f"\nbest test IoU {best_iou:.4f} @ epoch {best_epoch}")
print(f"history: {tracker.json_path}")
plot_history_live(tracker.history, title="ConvLSTM — final")


## Финальное превью на test-сэмпле

In [ ]:
for i in range(min(3, len(test_ds))):
    preview_prediction(
        model,
        test_ds,
        device,
        idx=i,
        title=f"test[{i}] — best IoU {best_iou:.4f} @ ep {best_epoch}",
    )
